# FOF / 01: futures data — Databento cost, download, NinjaTrader export, recorder, checks C1–C5

Protocol: [`001_claude_fof01_data_plan.md`](../../discussions/Futures_Order_Flow/001_claude_fof01_data_plan.md).

**No return statistics are computed here.** The notebook prices and buys Databento data within the budget, normalises the NinjaTrader export and the recorder files, and runs the data checks:

| Check | Pass |
|---|---|
| C1 Coverage | every CME trading day of each downloaded Databento month present |
| C2 Time zone | NinjaTrader vs Databento per-minute volume on the bridge (Nov–Dec 2025): best lag 0 min |
| C3 Signing | NinjaTrader quote/tick-rule sign = exchange aggressor on ≥ 95% of matched bridge volume |
| C4 Unsigned | ≤ 2% of volume unsigned, per source |
| C5 Roll | same roll trade dates in NinjaTrader and Databento on the bridge |

Holdout A (NinjaTrader 2026) and Holdout B (recorder) are reported as **day counts only**.

Run order: once with `DOWNLOAD = False` (prices only, nothing billed), then choose `REQUEST_ORDER` and run with `DOWNLOAD = True`. `FOF_SYNTHETIC=1` runs everything offline on synthetic data in a temporary folder.

In [ ]:
import json, os, sys, tempfile
from pathlib import Path
from types import SimpleNamespace
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SYNTHETIC = os.environ.get("FOF_SYNTHETIC") == "1"
if SYNTHETIC:                                   # never touch the real market-data folder in a smoke run
    os.environ["EQUITY_TRADING_DIR"] = tempfile.mkdtemp(prefix="fof01_")

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "equity_trading").exists())
sys.path.insert(0, str(ROOT / "src"))
from equity_trading import __version__
from equity_trading import futures as F
from equity_trading.data import data_dir, vendor_dir

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e6e3", "grid.linewidth": 0.6, "axes.titleweight": "bold", "font.size": 9})
OUT = ROOT / "notebooks" / "Futures_Order_Flow" / ("01_synthetic" if SYNTHETIC else "01")
OUT.mkdir(parents=True, exist_ok=True)

# ---- settings (FOF 001 §2) ----------------------------------------------------------------------------
SPEND_ORDER = [("NQ", 2025), ("ES", 2025), ("NQ", 2024), ("ES", 2024), ("NQ", 2023), ("ES", 2023),
               ("YM", 2025), ("YM", 2024), ("YM", 2023)]
REQUEST_ORDER = SPEND_ORDER          # after reading the cost table, cut this to what you want to buy
DOWNLOAD = False                     # True bills the Databento account (stops at F.BUDGET_USD)
BUDGET = F.BUDGET_USD
NT_EXPORT_TZ = "UTC"                 # NinjaTrader's General-options time zone at export time
BRIDGE = (pd.Timestamp("2025-11-01", tz="UTC"), F.HOLDOUT_START)
C3_MIN_AGREEMENT, C4_MAX_UNSIGNED = 0.95, 0.02
print(f"equity_trading {__version__}, futures {F.FUTURES_VERSION}; data dir {data_dir()}; synthetic={SYNTHETIC}")

## 0. Synthetic inputs (smoke run only)

One NQ tick path from 2025-11-03 to mid-January 2026, rolling from `NQ 12-25` to `NQ 03-26` in December. Databento gets the pre-2026 part with true sides; NinjaTrader gets the whole path as a five-field export; the recorder gets the 2026 part.

In [ ]:
class FakeClient:
    # Synthetic stand-in: quotes a fixed cost per month and never downloads.
    def __init__(self, cost):
        self.metadata = SimpleNamespace(get_cost=lambda **q: cost, get_billable_size=lambda **q: int(2e9))
        self.timeseries = SimpleNamespace(get_range=lambda **q: (_ for _ in ()).throw(RuntimeError("synthetic")))

if SYNTHETIC:
    path = F.synthetic_ticks("2025-11-03", days=75, per_day=1500, roll_day=40, seed=7)
    db_front = F.stitch_front(path[path["ts"] < F.HOLDOUT_START])        # Databento .v.0 is front-only
    F.write_months(db_front.assign(source="databento"), "databento", "NQ")
    nt_dir = vendor_dir("NinjaTrader") / "export" / "NQ"
    nt_dir.mkdir(parents=True, exist_ok=True)
    for c, part in path.groupby("contract"):
        (nt_dir / f"{c}.Last.txt").write_text("\n".join(F.to_nt_lines(part, NT_EXPORT_TZ)) + "\n")
    rec = path[path["ts"] >= F.HOLDOUT_START]
    rec_dir = vendor_dir("NinjaTrader") / "recorder"
    rec_dir.mkdir(parents=True, exist_ok=True)
    lines = ["ts_utc,instrument,price,size,bid,ask"] + [
        f"{t.strftime('%Y-%m-%dT%H:%M:%S.%f')}0Z,{c},{p},{s},{b},{a}"
        for t, c, p, s, b, a in zip(rec["ts"], rec["contract"], rec["price"], rec["size"], rec["bid"], rec["ask"])]
    (rec_dir / "NQ_synthetic.csv").write_text("\n".join(lines) + "\n")
    REQUEST_ORDER = [("NQ", 2025)]
    print(f"synthetic: {len(path):,} ticks, {path['contract'].nunique()} contracts")

## 1. Databento prices (free metadata calls; nothing is billed)

`cum_cost_usd` adds the months in `SPEND_ORDER` that are not cached yet. Spend so far comes from the ledger (`Databento/ledger.csv`). Requests that would take the total above the budget are marked `over_budget`.

In [ ]:
cl = FakeClient(cost=4.0) if SYNTHETIC else F.client()
plan = F.plan_requests(SPEND_ORDER)
costs = F.estimate_costs(cl, plan)
spent = F.spent_usd()
costs["over_budget"] = spent + costs["cum_cost_usd"] > BUDGET
costs.to_csv(OUT / "01_cost_estimate.csv", index=False, float_format="%.4f")
by_year = (costs.assign(year=costs["month"].str[:4])
                .groupby(["root", "year"], sort=False)
                .agg(months=("month", "size"), cost_usd=("cost_usd", "sum"), gb=("gb", "sum"),
                     cached=("cached", "sum"), over_budget=("over_budget", "any")))
by_year["cum_cost_usd"] = by_year["cost_usd"].cumsum()
print(f"ledger spend so far: ${spent:.2f}; budget ${BUDGET:.2f}; remaining ${BUDGET - spent:.2f}")
display(by_year.round(2))

## 2. Download (only with `DOWNLOAD = True`)

Months are bought in `REQUEST_ORDER`; the run stops before the first month that would exceed the budget. Raw DBN files are kept under `Databento/raw/`, so a month is never bought twice. The 2026 months are refused in code.

In [ ]:
if DOWNLOAD and not SYNTHETIC:
    log = F.download(cl, F.plan_requests(REQUEST_ORDER), budget=BUDGET)
    log.to_csv(OUT / "01_download_log.csv", index=False)
    display(log)
    print(f"ledger spend now: ${F.spent_usd():.2f}")
else:
    print("DOWNLOAD is False: nothing requested." if not SYNTHETIC else "synthetic: Databento months written in §0.")

## 3. C1 and C4 on the Databento months present

Per CME trade date: ticks, volume, unsigned share. Only timestamps, sizes and signs are read. Expected days come from the CME equity calendar.

In [ ]:
def months_present(source, root):
    ms = sorted(p.stem for p in F.cache_dir(source, root).glob("*.parquet"))
    return [m for m in ms if pd.Timestamp(m + "-01", tz="UTC") < F.HOLDOUT_START]

C1, C4, COV = {}, {}, {}
for root in sorted({r for r, _ in SPEND_ORDER}):
    ms = months_present("databento", root)
    if not ms:
        continue
    parts = []
    for m in ms:
        s = pd.Timestamp(m + "-01", tz="UTC")
        e = s + pd.offsets.MonthBegin(1)
        parts.append(F.coverage(F.load_ticks(root, s, e, "databento", columns=["ts", "size", "sign"])))
    cov = pd.concat(parts).groupby(level=0).sum()
    cov["unsigned_share"] = cov["unsigned_volume"] / cov["volume"]
    expected = [d for m in ms for d in F.cme_trading_days(m + "-01", pd.Timestamp(m + "-01") + pd.offsets.MonthBegin(1))]
    missing = F.missing_trade_dates(cov, expected)
    unsigned = float(cov["unsigned_volume"].sum() / cov["volume"].sum())
    COV[root] = cov
    C1[root] = {"months": len(ms), "first": ms[0], "last": ms[-1], "trade_dates": len(cov),
                "expected": len(expected), "missing": [str(d) for d in missing], "pass": not missing}
    C4[f"databento_{root}"] = {"unsigned_share": unsigned, "pass": unsigned <= C4_MAX_UNSIGNED}
    cov.to_csv(OUT / f"01_coverage_databento_{root}.csv")
display(pd.DataFrame(C1).T)
if COV:
    fig, ax = plt.subplots(len(COV), 1, figsize=(9, 2.2 * len(COV)), squeeze=False)
    for a, (root, cov) in zip(ax[:, 0], COV.items()):
        a.bar(pd.to_datetime(cov.index), cov["volume"] / 1e3, width=1.0, color="#2a78d6")
        a.set_title(f"{root}: Databento volume per CME trade date (thousand contracts)")
    fig.tight_layout(); fig.savefig(OUT / "01_coverage_databento.png"); plt.show()

## 4. NinjaTrader export: normalise, then C2, C3, C4, C5 on the bridge

Each `NinjaTrader/export/<ROOT>/` folder is loaded (signed by the quote rule, stitched by prior-day volume) and written to the monthly cache. The bridge comparison uses Nov–Dec 2025 only. 2026 is written to disk but not read here except as day counts.

In [ ]:
NT_ROOTS = []
export_root = vendor_dir("NinjaTrader") / "export"
for folder in sorted(p for p in export_root.glob("*") if p.is_dir() and any(p.glob("*.txt"))):
    nt = F.load_nt_root(folder, NT_EXPORT_TZ)
    F.write_months(nt, "ninjatrader", folder.name)
    NT_ROOTS.append(folder.name)
    print(f"{folder.name}: {len(nt):,} ticks, {nt['contract'].nunique()} contracts, "
          f"{F.cme_trade_date(nt['ts']).nunique()} trade dates")
    del nt
if not NT_ROOTS:
    print(f"no NinjaTrader exports under {export_root}")

In [ ]:
C2, C3, C5 = {}, {}, {}
for root in NT_ROOTS:
    nt = F.load_ticks(root, *BRIDGE, "ninjatrader")
    db = F.load_ticks(root, *BRIDGE, "databento")
    unsigned = float((nt["size"] * (nt["sign"] == 0)).sum() / max(nt["size"].sum(), 1))
    C4[f"ninjatrader_{root}"] = {"unsigned_share": unsigned, "pass": unsigned <= C4_MAX_UNSIGNED}
    if nt.empty or db.empty:
        C2[root] = C3[root] = C5[root] = {"status": "NO_OVERLAP", "nt_ticks": len(nt), "db_ticks": len(db)}
        continue
    lag, corr = F.best_lag_minutes(nt, db)
    C2[root] = {"best_lag_min": lag, "corr": round(corr, 4), "pass": lag == 0}
    agree = F.sign_agreement(nt, db)
    agree["pass"] = bool(agree["agreement"] >= C3_MIN_AGREEMENT) if agree["match_rate"] >= 0.5 else False
    agree["status"] = "OK" if agree["match_rate"] >= 0.5 else "INSUFFICIENT_MATCH"
    C3[root] = agree
    by_method = nt.groupby("sign_method")["size"].sum() / nt["size"].sum()
    C3[root]["nt_volume_by_method"] = by_method.round(4).to_dict()
    r_nt, r_db = F.roll_dates(nt), F.roll_dates(db)
    C5[root] = {"nt_rolls": [str(d) for d in r_nt["trade_date"]], "db_rolls": [str(d) for d in r_db["trade_date"]]}
    C5[root]["pass"] = C5[root]["nt_rolls"] == C5[root]["db_rolls"]
    del nt, db
for name, d in (("C2 time zone", C2), ("C3 signing", C3), ("C5 roll", C5)):
    if d:
        print(name); display(pd.DataFrame(d).T)
print("C4 unsigned"); display(pd.DataFrame(C4).T)

## 5. Holdouts: day counts only

Holdout A is the NinjaTrader export from 2026-01-01; Holdout B is the recorder. Recorder CSVs in `NinjaTrader/recorder/` are normalised into the monthly cache first. Only timestamps are read.

In [ ]:
rec_files = sorted((vendor_dir("NinjaTrader") / "recorder").glob("*.csv"))
for root, ticks in F.read_recorder(rec_files).items():
    F.write_months(ticks, "recorder", root)
HOLD = []
for source in ("ninjatrader", "recorder"):
    for folder in sorted((vendor_dir(F.SOURCES[source][0]) / F.SOURCES[source][1]).glob("*")):
        if folder.is_dir():
            c = F.holdout_day_counts(folder.name, source)
            if not c.empty:
                HOLD.append(c.assign(source=source, root=folder.name))
HOLD = pd.concat(HOLD, ignore_index=True) if HOLD else pd.DataFrame(columns=["month", "trade_dates", "ticks", "source", "root"])
HOLD.to_csv(OUT / "01_holdout_day_counts.csv", index=False)
display(HOLD.pivot_table(index="month", columns=["source", "root"], values="trade_dates", aggfunc="sum")
        if not HOLD.empty else "no holdout data yet")

## 6. Decision

In [ ]:
def allpass(d):
    return bool(d) and all(v.get("pass", False) for v in d.values())

decision = {
    "version": f"FOF-01-{F.FUTURES_VERSION}",
    "synthetic": SYNTHETIC,
    "ledger_spend_usd": round(F.spent_usd(), 2),
    "budget_usd": BUDGET,
    "C1_coverage": C1, "C2_time_zone": C2, "C3_signing": C3, "C4_unsigned": C4, "C5_roll": C5,
    "gates": {"C1": allpass(C1), "C2": allpass(C2), "C3": allpass(C3), "C4": allpass(C4), "C5": allpass(C5)},
    "holdout_months": HOLD.groupby("source")["month"].nunique().to_dict() if not HOLD.empty else {},
}
decision["data_ready_for_FOF_002"] = all(decision["gates"].values())
(OUT / "01_decision.json").write_text(json.dumps(decision, indent=1, default=str))
print(json.dumps(decision["gates"], indent=1), "\ndata_ready_for_FOF_002:", decision["data_ready_for_FOF_002"])